# Frozen comparison sweep

Original setup, tests, resource probes, smoke and full-training cells are retained as the historical GPU training record. All training remains Colab-only. The current comparison entry point uses CPU only; no GPU allocation is needed. Outputs and checkpoints remain in private Google Drive. Existing seeds, frozen data, sentence-prefix budgets and per-budget vocabularies are preserved.

## CPU Comparison Only: Gutenberg and Inaugural Full Runs

Select a Colab runtime with hardware accelerator CPU and run only Cell 8, the final code cell. Do not use Run all or rerun the original setup, tests, resource probes, smoke or full-training cells. Comparison uses only the full/runs artifacts for Gutenberg and Inaugural at 20%, 40%, 60%, 80% and 100%; smoke/runs is never used. Saved neural checkpoints are loaded on CPU for scoring, without neural training. Statistical baselines are fitted on the exact matching training snapshots. Metrics, predictions, rare-sequence scores, charts, event verification and artifact manifests are captured. Existing partial metrics are backed up before replacement, and each attempt has separate logs under comparison_attempts. The original Drive run and PyTorch release must be accessible; CPU/CUDA build suffixes may differ and both versions are recorded. A CPU-compatible runtime copy changes only the comparison CUDA guard while retaining the Colab-only check; the frozen source bundle and trained models are untouched. Other corpora are untouched. Missing full-run checkpoints stop execution before scoring, rather than triggering retraining or silently skipping models.

In [ ]:
import os
import sys
import hashlib
import json
import zipfile
import subprocess
import shutil
from pathlib import Path

assert os.environ.get('COLAB_RELEASE_TAG'), 'Run this notebook in Google Colab only'
import torch
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before continuing'
from google.colab import drive, files
drive.mount('/content/drive')
WORKSPACE = Path('/content/nlp_comparison_frozen_v3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP/comparison_sweep_20261102_v3')
TESTS_PASSED = False
SMOKE_PASSED = False
RESOURCE_PREFLIGHT_PASSED = False
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
WORKSPACE.mkdir(parents=True, exist_ok=True)
EXPECTED_BUNDLE_SHA256 = '2ae43095c6e7f0fc95468b13a40c9d49b9004c7579bb5b961b2f7f6e60bbfff8'
bundle = DRIVE_ROOT / 'comparison_sweep_frozen_v3.zip'
local_bundle = Path('/content/comparison_sweep_frozen_v3.zip')
if not bundle.is_file() and local_bundle.is_file():
    shutil.copyfile(local_bundle, bundle)
if not bundle.is_file():
    uploaded = files.upload()
    assert set(uploaded) == {'comparison_sweep_frozen_v3.zip'}, 'Upload the frozen ZIP bundle only'
    bundle.write_bytes(uploaded['comparison_sweep_frozen_v3.zip'])
    del uploaded
assert hashlib.sha256(bundle.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256, 'Bundle checksum mismatch'
with zipfile.ZipFile(bundle) as archive:
    bundle_manifest = json.loads(archive.read('bundle_manifest.json'))
    assert set(archive.namelist()) == set(bundle_manifest['files']) | {'bundle_manifest.json'}
    for name, expected in bundle_manifest['files'].items():
        target = (WORKSPACE / name).resolve()
        assert target.is_relative_to(WORKSPACE.resolve()), 'Unsafe archive member'
        assert hashlib.sha256(archive.read(name)).hexdigest() == expected, name
    archive.extractall(WORKSPACE)
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'src/requirements.txt'], check=True)
import yaml
from run import verify_prepared_inputs
for phase in ('smoke', 'full'):
    runtime_configs = WORKSPACE / 'runtime_configs' / phase
    runtime_configs.mkdir(parents=True, exist_ok=True)
    for config_path in sorted((WORKSPACE / 'snapshot/configs').glob('*.yaml')):
        config = yaml.safe_load(config_path.read_text())
        config['data']['output'] = str(WORKSPACE / 'snapshot/processed')
        config['data']['prepared_manifest'] = str(WORKSPACE / 'snapshot/processed' / config['data']['corpus'] / 'prepared_manifest.json')
        config['output_dir'] = str(DRIVE_ROOT / phase / 'runs')
        (runtime_configs / config_path.name).write_text(yaml.safe_dump(config, sort_keys=False))
        if bundle_manifest['corpora'][config['data']['corpus']]['status'] == 'prepared':
            verify_prepared_inputs(config)
    manifest_name = 'mega-smoke.yaml'
    if phase == 'full':
        manifest_name = 'mega-run.yaml'
    manifest = yaml.safe_load((WORKSPACE / 'snapshot' / manifest_name).read_text())
    manifest['run'] = [str(runtime_configs / Path(name).name) for name in manifest['run']]
    (runtime_configs / manifest_name).write_text(yaml.safe_dump(manifest))
(DRIVE_ROOT / 'bundle_manifest.json').write_text(json.dumps(bundle_manifest, indent=2))
with (DRIVE_ROOT / 'colab_environment.txt').open('w') as package_log:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=package_log, check=True)
environment = dict(os.environ, PYTHONPATH=str(WORKSPACE / 'src'))
print('GPU:', torch.cuda.get_device_name(0))
print('Verified packaged files:', len(bundle_manifest['files']))
print('Full sweep ready:', bundle_manifest['full_sweep_ready'])

In [ ]:
assert os.environ.get('COLAB_RELEASE_TAG') and torch.cuda.is_available()
with (DRIVE_ROOT / 'regression_tests.log').open('w') as log:
    completed = subprocess.run([sys.executable, '-u', '-m', 'unittest', 'discover', '-s', 'src/tests'], env=environment, stdout=log, stderr=subprocess.STDOUT)
print((DRIVE_ROOT / 'regression_tests.log').read_text()[-16000:])
assert completed.returncode == 0, 'Regression tests failed; do not train the sweep'
TESTS_PASSED = True

In [ ]:
assert TESTS_PASSED
resource_probe = r'''
import gc
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path
import torch
import yaml
from lm.data import Vocab
from lm.neural import build_model
from lm.neural.trainer import Trainer, TrainConfig, set_seed
from lm.preparation import atomic_json
from lm.tokenization import get_tokenizer
from run import verify_prepared_inputs
assert os.environ.get('COLAB_RELEASE_TAG') and torch.cuda.is_available()
workspace, output_root = map(Path, sys.argv[1:3])
manifest = yaml.safe_load((workspace / 'runtime_configs/full/mega-run.yaml').read_text())
probes = []
for config_name in manifest['run']:
    config = yaml.safe_load(Path(config_name).read_text())
    prepared = verify_prepared_inputs(config)
    corpus = config['data']['corpus']
    tokenizer = get_tokenizer(config['data'].get('tokenizer', 'space'))
    counts = Counter()
    train_path = Path(config['data']['output']) / corpus / 'train.txt'
    with train_path.open() as handle:
        for line in handle:
            words = tokenizer.tokenize(line)
            if config['data'].get('lowercase', True):
                words = [word.lower() for word in words]
            counts.update(words)
    options = config.get('vocab', {})
    vocab = Vocab.from_counts(counts, min_freq=options.get('min_freq', 2), max_size=options.get('max_size'))
    longest = max(split['longest_sentence_tokens'] for split in prepared['counts'].values()) + 1
    batch_size = config['training']['batch_size']
    train_config = TrainConfig(**{key: value for key, value in config['training'].items() if key != 'batch_size'})
    for spec in config['models']:
        set_seed(config['seeds'][0])
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        model = build_model(spec['name'], len(vocab), vocab.pad_id, **spec.get('params', {}))
        trainer = Trainer(model, train_config, vocab.pad_id, torch.device('cuda'), allowed_target_ids=vocab.allowed_target_ids)
        inputs = torch.full((batch_size, 64), vocab.unk_id, dtype=torch.long)
        targets = inputs.clone()
        loss, tokens = trainer._batch_loss(inputs, targets)
        assert torch.isfinite(loss)
        trainer.optimizer.zero_grad()
        (loss / tokens).backward()
        trainer.optimizer.step()
        model.eval()
        with torch.no_grad():
            validation = torch.full((1, longest), vocab.unk_id, dtype=torch.long)
            validation_loss, _ = trainer._batch_loss(validation, validation)
            assert torch.isfinite(validation_loss)
        torch.cuda.synchronize()
        probe = dict(corpus=corpus, model=spec['name'], params=spec.get('params', {}), vocab_size=len(vocab), longest_sequence=longest,
                     peak_reserved_gib=torch.cuda.max_memory_reserved() / 1024**3, seconds=time.perf_counter() - started)
        probes.append(probe)
        print(json.dumps(probe), flush=True)
        atomic_json(output_root / 'resource_preflight.json', dict(passed=False, probes=probes, kind='disposable_single_batch_probes'))
        del trainer, model, loss, inputs, targets, validation, validation_loss
        gc.collect()
        torch.cuda.empty_cache()
    del vocab, counts
assert len(probes) == 18
atomic_json(output_root / 'resource_preflight.json', dict(passed=True, probes=probes, kind='disposable_single_batch_probes'))
'''
compile(resource_probe, '<colab-resource-probe>', 'exec')
with (DRIVE_ROOT / 'resource_preflight.log').open('w') as log:
    completed = subprocess.run([sys.executable, '-u', '-c', resource_probe, str(WORKSPACE), str(DRIVE_ROOT)], env=environment, stdout=log, stderr=subprocess.STDOUT)
print((DRIVE_ROOT / 'resource_preflight.log').read_text()[-16000:])
assert completed.returncode == 0, 'Resource preflight failed; do not start the full sweep'
assert json.loads((DRIVE_ROOT / 'resource_preflight.json').read_text())['passed']
RESOURCE_PREFLIGHT_PASSED = True

In [ ]:
assert TESTS_PASSED
command = [sys.executable, '-u', 'src/run.py', '--full', str(WORKSPACE / 'runtime_configs/smoke/mega-smoke.yaml'), '--parallelism', '1', '--resume']
with (DRIVE_ROOT / 'smoke.log').open('a') as log:
    completed = subprocess.run(command, env=environment, stdout=log, stderr=subprocess.STDOUT)
print((DRIVE_ROOT / 'smoke.log').read_text()[-16000:])
assert completed.returncode == 0, 'Smoke failed; inspect durable smoke.log'
SMOKE_PASSED = True

In [ ]:
assert TESTS_PASSED and SMOKE_PASSED
import csv
from lm.preparation import atomic_json, file_sha256
from run import verify_prepared_inputs, verify_model_artifacts, expected_variants
from model_comparison import verify_scoring_events

smoke_config_path = WORKSPACE / 'runtime_configs/smoke/inaugural.yaml'
smoke_config = yaml.safe_load(smoke_config_path.read_text())
verify_prepared_inputs(smoke_config)
missing = verify_model_artifacts(smoke_config_path, smoke_config, [20, 40], check_comparison=True)
assert not missing, missing
run_root = Path(smoke_config['output_dir']) / smoke_config['name']
metrics_root = run_root / 'metrics'
inventory_path = metrics_root / 'artifact_manifest.json'
inventory = json.loads(inventory_path.read_text())
assert inventory['config_sha256'] == file_sha256(smoke_config_path)
observed = {target.relative_to(run_root).as_posix() for target in run_root.rglob('*') if target.is_file() and target != inventory_path}
assert observed == set(inventory['files']), 'Artifact inventory is incomplete'
for name, expected in inventory['files'].items():
    target = (run_root / name).resolve()
    assert target.is_relative_to(run_root.resolve()), 'Unsafe artifact path'
    assert file_sha256(target) == expected, name
reconstructed = verify_scoring_events(metrics_root / 'metrics.csv', metrics_root / 'scoring_events.jsonl.gz')
assert reconstructed == json.loads((metrics_root / 'event_verification.json').read_text())
assert reconstructed['passed'] and reconstructed['conditions'] == 90
weights = []
for percent in (20, 40):
    directory = run_root / f'train_{percent}pct'
    assert (directory / 'count_models.pkl.gz').is_file()
    for folder, model_name, _ in expected_variants(smoke_config):
        for seed in smoke_config['seeds']:
            weight = directory / 'models' / folder / f'{model_name}_seed{seed}.pt'
            assert weight.is_file(), str(weight)
            weights.append(weight)
assert len(weights) == 18
for weight in weights:
    history_path = weight.with_name(weight.stem + '_history.csv')
    with history_path.open(newline='') as history:
        assert list(csv.DictReader(history)), str(history_path)
smoke_verification = dict(reconstructed, artifact_files=len(inventory['files']), neural_checkpoints=len(weights), training_histories=len(weights), statistical_conditions=12)
atomic_json(DRIVE_ROOT / 'smoke_verification.json', smoke_verification)
print(json.dumps(smoke_verification, indent=2))

In [ ]:
assert TESTS_PASSED and SMOKE_PASSED and RESOURCE_PREFLIGHT_PASSED
assert bundle_manifest['full_sweep_ready'], 'Full sweep blocked: incomplete prepared inputs or unaccepted audit policy'
assert json.loads((DRIVE_ROOT / 'smoke_verification.json').read_text())['passed']
assert json.loads((DRIVE_ROOT / 'resource_preflight.json').read_text())['passed']
command = [sys.executable, '-u', 'src/run.py', '--full', str(WORKSPACE / 'runtime_configs/full/mega-run.yaml'), '--parallelism', '1', '--resume']
print('Starting full six-corpus sweep; detailed log:', DRIVE_ROOT / 'full.log', flush=True)
with (DRIVE_ROOT / 'full.log').open('a') as log:
    with subprocess.Popen(command, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            if any(marker in line for marker in ('Full run:', 'FULL PIPELINE:', 'Verified frozen', 'Train all configured', 'Verified model checkpoints', 'Compare all percentage', 'Full pipeline complete')):
                print(line.rstrip(), flush=True)
        returncode = process.wait()
print((DRIVE_ROOT / 'full.log').read_text()[-16000:])
assert returncode == 0, 'Full sweep failed; inspect durable full.log before resuming'

In [ ]:
import os
import sys
import ast
import json
import hashlib
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

assert os.environ.get('COLAB_RELEASE_TAG'), 'Comparison runs in Google Colab only'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import torch
from packaging.version import Version
from google.colab import drive
drive.mount('/content/drive')

WORKSPACE = Path('/content/nlp_comparison_frozen_v3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP/comparison_sweep_20261102_v3')
COMPARISON_CORPORA = ['gutenberg', 'inaugural']
COMPARISON_PERCENTAGES = [20, 40, 60, 80, 100]
prior_models = DRIVE_ROOT / 'full/runs/gutenberg_neural/train_20pct/models'
prior_result_path = next(prior_models.rglob('*_result.json'), None)
assert prior_result_path is not None, 'Original full trained run is not accessible in this Drive'
prior_result = json.loads(prior_result_path.read_text())
prior_torch = prior_result['signature_payload']['torch_version']
assert Version(str(torch.__version__)).public == Version(prior_torch).public, f'Restore PyTorch release {Version(prior_torch).public}; current runtime is {torch.__version__}'

if not (WORKSPACE / 'snapshot/configs/gutenberg.yaml').is_file():
    bundle = DRIVE_ROOT / 'comparison_sweep_frozen_v3.zip'
    expected_sha = '2ae43095c6e7f0fc95468b13a40c9d49b9004c7579bb5b961b2f7f6e60bbfff8'
    assert hashlib.sha256(bundle.read_bytes()).hexdigest() == expected_sha, 'Original V3 bundle checksum mismatch'
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle) as archive:
        archive.extractall(WORKSPACE)
os.chdir(WORKSPACE)
attempt_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
attempt_root = DRIVE_ROOT / 'comparison_attempts' / attempt_id
attempt_root.mkdir(parents=True, exist_ok=False)
with (attempt_root / 'dependencies.log').open('w') as package_log:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'src/requirements.txt'], stdout=package_log, stderr=subprocess.STDOUT, check=True)
import yaml
sys.path.insert(0, str(WORKSPACE / 'src'))
from run import verify_model_artifacts
config_root = WORKSPACE / 'runtime_configs/comparison_cpu'
config_root.mkdir(parents=True, exist_ok=True)


def cpu_comparison_source(source):
    tree = ast.parse(source)
    run_function = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'run')
    original_guard = ast.parse('not os.environ.get("COLAB_RELEASE_TAG") or not torch.cuda.is_available()', mode='eval').body
    guards = [node for node in ast.walk(run_function) if isinstance(node, ast.If) and ast.dump(node.test) == ast.dump(original_guard)]
    assert len(guards) == 1, 'Frozen comparison CUDA guard changed; stop rather than bypass execution checks'
    guard = guards[0]
    guard.test = ast.parse('not os.environ.get("COLAB_RELEASE_TAG")', mode='eval').body
    guard.body[0].exc.args[0] = ast.Constant('This experiment may only fit count tables in a Colab runtime')
    return ast.unparse(ast.fix_missing_locations(tree)) + '\n'


original_comparison_source = (WORKSPACE / 'src/model_comparison.py').read_text()
comparison_source = cpu_comparison_source(original_comparison_source)
cpu_script = config_root / 'model_comparison_cpu.py'
cpu_script.write_text(comparison_source)
(attempt_root / 'model_comparison_cpu.py').write_text(comparison_source)
selected_configs = []
for corpus in COMPARISON_CORPORA:
    config = yaml.safe_load((WORKSPACE / 'snapshot/configs' / f'{corpus}.yaml').read_text())
    config['data']['output'] = str(WORKSPACE / 'snapshot/processed')
    config['data']['prepared_manifest'] = str(WORKSPACE / 'snapshot/processed' / corpus / 'prepared_manifest.json')
    config['output_dir'] = str(DRIVE_ROOT / 'full/runs')
    config['device'] = 'cpu'
    config_path = config_root / f'{corpus}.yaml'
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    missing = verify_model_artifacts(config_path, config, COMPARISON_PERCENTAGES)
    assert not missing, f'Comparison requires existing full-run checkpoints; no retraining will be attempted: {missing[:10]}'
    selected_configs.append((corpus, config))
for corpus, config in selected_configs:
    metrics_root = Path(config['output_dir']) / config['name'] / 'metrics'
    if metrics_root.is_dir() and any(metrics_root.iterdir()):
        backup_root = attempt_root / 'previous_metrics' / corpus
        shutil.copytree(metrics_root, backup_root)
        print('Preserved existing metrics:', backup_root, flush=True)
with (attempt_root / 'environment.txt').open('w') as package_log:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=package_log, check=True)
(attempt_root / 'comparison_plan.json').write_text(json.dumps({
    'corpora': COMPARISON_CORPORA,
    'percentages': COMPARISON_PERCENTAGES,
    'run_phase': 'full',
    'smoke_runs': 'not_run',
    'device': 'cpu',
    'cuda_visible_devices': '',
    'neural_training': False,
    'statistical_baselines': 'fit_and_score_exact_saved_training_snapshots',
    'startup_tests_smoke_resource_probes': 'not_run',
    'training_torch_version': prior_torch,
    'torch_version': str(torch.__version__),
    'original_comparison_sha256': hashlib.sha256(original_comparison_source.encode()).hexdigest(),
    'cpu_comparison_sha256': hashlib.sha256(comparison_source.encode()).hexdigest(),
    'runtime_adaptation': 'comparison CUDA availability guard removed; Colab-only guard retained',
}, indent=2))

comparison_program = r'''
import os
import json
import subprocess
import sys
from pathlib import Path

assert os.environ.get('COLAB_RELEASE_TAG'), 'Comparison runs in Google Colab only'
assert os.environ.get('CUDA_VISIBLE_DEVICES') == '', 'GPU visibility must be disabled for CPU-only comparison'


def compare_corpora(config_root, corpora):
    for corpus in corpora:
        print(f'COMPARISON ONLY {corpus}: full training runs, CPU', flush=True)
        subprocess.run([sys.executable, '-u', str(config_root / 'model_comparison_cpu.py'), corpus, '--config', str(config_root / f'{corpus}.yaml'), '--device', 'cpu'], check=True)
        print(f'COMPARISON COMPLETE {corpus}', flush=True)


compare_corpora(Path(sys.argv[1]), json.loads(sys.argv[2]))
print('COMPARISON COMPLETE ALL REQUESTED CORPORA', flush=True)
'''
(attempt_root / 'comparison_program.py').write_text(comparison_program)
environment = dict(os.environ, PYTHONPATH=str(WORKSPACE / 'src'), CUDA_VISIBLE_DEVICES='')
command = [sys.executable, '-u', '-c', comparison_program, str(config_root), json.dumps(COMPARISON_CORPORA)]
log_path = attempt_root / 'comparison.log'
print('CPU comparison only: Gutenberg and Inaugural full training runs, all five budgets; no smoke runs or neural training. Log:', log_path, flush=True)
with log_path.open('w') as log:
    with subprocess.Popen(command, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            if any(marker in line for marker in ('COMPARISON ', '== ', 'Corpus:', 'Vocabulary:', ' / train:', ' / valid:', ' / test:', ' / rare ', 'Wrote ')) or 'progress' in line.lower():
                print(line.rstrip(), flush=True)
        returncode = process.wait()
if returncode:
    print(log_path.read_text()[-16000:])
assert returncode == 0, 'Comparison stopped; inspect its attempt log. Neural retraining is not needed.'